> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `814281740296490` · task `t01_raw_to_bronze_initial` · notebook `01_raw_to_bronze` · started 2026-10-11 12:56 UTC · result **SUCCESS**  
> Parameters: `catalog=workspace`, `drift_mode=evolve`, `load_type=all`, `mode=incremental`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 01 · Raw → Bronze
Loads landing batches (JSONL + `_manifest.json`) into the Bronze Delta tables using **explicit StructType
schemas** (no inference). Invalid records go to `so_bronze.quarantine_records`, schema drift is recorded in
`so_ops.schema_drift_events`, and every file processed writes a row to `so_ops.pipeline_execution_logs`.

| Widget | Standard incremental run | Backfill |
|---|---|---|
| `mode` | `incremental`: every batch not yet loaded | `backfill`: reload the selected batches |
| `load_type` | `all` | `all`, `full` or `incremental` |
| `batch_ids` | empty | e.g. `incr_20260925T155859Z,full_2025-01-01_2025-04-01` |
| `start_date` / `end_date` | empty | extraction-date range, `YYYY-MM-DD`, inclusive |
| `landing_path` | empty | an explicit batch folder, e.g. `/Volumes/workspace/so_raw/landing/incremental/incr_...` |
| `drift_mode` | `evolve` | `evolve` adds new columns (mergeSchema); `rescue` keeps them in `_rescued_data` |

In [1]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.dropdown("load_type", "all", ["all", "full", "incremental"])
dbutils.widgets.text("batch_ids", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("landing_path", "")
dbutils.widgets.dropdown("drift_mode", "evolve", ["evolve", "rescue"])

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.bronze import run_raw_to_bronze

cfg = build_config()
audit = run_raw_to_bronze(
    spark, cfg,
    mode=widget("mode"),
    load_type=widget("load_type"),
    batch_ids=widget("batch_ids") or None,
    start_date=widget("start_date") or None,
    end_date=widget("end_date") or None,
    landing_path=widget("landing_path") or None,
    fail_on_error=True,
)

Raw-to-Bronze: 5 batch(es) selected, mode=incremental
[SUCCESS] Raw-to-Bronze | questions                          | read=2738 ins=2738 upd=0 del=0 quar=0 | 19.9s
[SUCCESS] Raw-to-Bronze | answers                            | read=3384 ins=3384 upd=0 del=0 quar=0 | 9.9s
[SUCCESS] Raw-to-Bronze | questions                          | read=4548 ins=4548 upd=0 del=0 quar=0 | 9.6s
[SUCCESS] Raw-to-Bronze | answers                            | read=5528 ins=5528 upd=0 del=0 quar=0 | 9.8s
[SUCCESS] Raw-to-Bronze | questions                          | read=35 ins=35 upd=0 del=0 quar=0 | 7.2s
[SUCCESS] Raw-to-Bronze | answers                            | read=59 ins=59 upd=0 del=0 quar=0 | 6.6s
[SUCCESS] Raw-to-Bronze | deleted_questions                  | read=0 ins=0 upd=0 del=0 quar=0 | 5.3s
[SUCCESS] Raw-to-Bronze | question_snapshots                 | read=100 ins=100 upd=0 del=0 quar=0 | 7.9s
[SUCCESS] Raw-to-Bronze | questions                          | read=305 ins=305 upd=0 del=0 quar=

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
77732aea-fab7-4e52-90c4-cacb8321cc87,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,full,questions,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/questions.jsonl,workspace.so_bronze.questions,2026-10-11T12:56:11.847Z,2026-10-11T12:56:31.755Z,19.908026,SUCCESS,2738,2738,0,0,0,,2026-10-11T12:56:31.755Z
31100b83-fc43-4df3-8111-0984e032091f,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,full,answers,full_2025-04-01_2025-07-01,/Volumes/workspace/so_raw/landing/full/full_2025-04-01_2025-07-01/answers.jsonl,workspace.so_bronze.answers,2026-10-11T12:56:34.395Z,2026-10-11T12:56:44.276Z,9.88154,SUCCESS,3384,3384,0,0,0,,2026-10-11T12:56:44.276Z
e0dd0ca3-02eb-43b2-88b2-f97ba870b695,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,full,questions,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/questions.jsonl,workspace.so_bronze.questions,2026-10-11T12:56:52.110Z,2026-10-11T12:57:01.749Z,9.638438,SUCCESS,4548,4548,0,0,0,,2026-10-11T12:57:01.749Z
ad44193b-9a92-4800-974c-bd367a8969b1,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,full,answers,full_2025-01-01_2025-04-01,/Volumes/workspace/so_raw/landing/full/full_2025-01-01_2025-04-01/answers.jsonl,workspace.so_bronze.answers,2026-10-11T12:57:03.303Z,2026-10-11T12:57:13.061Z,9.758112,SUCCESS,5528,5528,0,0,0,,2026-10-11T12:57:13.061Z
f63e1004-276a-4827-ab24-93f96cc304e2,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,incremental,questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/questions.jsonl,workspace.so_bronze.questions,2026-10-11T12:57:19.730Z,2026-10-11T12:57:26.894Z,7.16387,SUCCESS,35,35,0,0,0,,2026-10-11T12:57:26.894Z
376c7fb5-ba8e-4083-8538-8f360b3bad0e,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,incremental,answers,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/answers.jsonl,workspace.so_bronze.answers,2026-10-11T12:57:29.131Z,2026-10-11T12:57:35.770Z,6.63907,SUCCESS,59,59,0,0,0,,2026-10-11T12:57:35.770Z
1ae400a8-b858-4b14-9fd7-9502cf09c3e7,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,incremental,deleted_questions,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/deleted_questions.jsonl,workspace.so_bronze.deleted_questions,2026-10-11T12:57:37.281Z,2026-10-11T12:57:42.578Z,5.296855,SUCCESS,0,0,0,0,0,,2026-10-11T12:57:42.578Z
8c95bcd5-2d5a-4ef1-ab9c-ebd9d4b88a21,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,incremental,question_snapshots,incr_20260925T155859Z,/Volumes/workspace/so_raw/landing/incremental/incr_20260925T155859Z/question_snapshots.jsonl,workspace.so_bronze.question_snapshots,2026-10-11T12:57:44.348Z,2026-10-11T12:57:52.213Z,7.865152,SUCCESS,100,100,0,0,0,,2026-10-11T12:57:52.213Z
e00ade5b-19de-42f7-b5c4-d95f40025a03,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,incremental,questions,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/questions.jsonl,workspace.so_bronze.questions,2026-10-11T12:57:58.084Z,2026-10-11T12:58:05.752Z,7.667964,SUCCESS,305,305,0,0,0,,2026-10-11T12:58:05.752Z
79482031-a7a5-46c9-aee0-c23693dbd526,raw_to_bronze-20261011T125609Z-3c944e,raw_to_bronze,Raw-to-Bronze,incremental,incremental,answers,incr_20261007T155914Z,/Volumes/workspace/so_raw/landing/incremental/incr_20261007T155914Z/answers.jsonl,workspace.so_bronze.answers,2026-10-11T12:58:07.213Z,2026-10-11T12:58:15.044Z,7.831347,SUCCES